# Sentiment Analysis on Amazon Reviews



## 1. Loading the Dataset


In [ ]:
import kagglehub
import pandas as pd
import nltk
import dataframe_image as dfi

# Download latest version
# path = kagglehub.dataset_download("kritanjalijain/amazon-reviews")

def save_snapshot(dataFrame, fileName):

    def truncate_text(text, max_length=80):
        if len(text) > max_length:
            return text[:max_length] + '...'
        return text

    df = dataFrame.loc[[2, 8, 10, 31]]
    styled_df = df.style.format({'text': truncate_text})

    # Apply additional custom styles
    styled_df = styled_df.hide(axis="index").set_table_styles([
        {'selector': 'tbody td',
        'props': [('background-color', '#222222'),  # Dark gray background
                ('color', '#ffffff'),             # White text
                ('border', '1px solid #444444')]},# Subtle border
        {'selector': 'thead th',
        'props': [('background-color', '#444444'),  # Lighter gray for headers
                ('color', '#ffffff'),             # White text
                ('border', '1px solid #666666')]} # Subtle border
    ])
        
    df.to_csv(f"{fileName}.csv", index=False)
    dfi.export(styled_df, f'{fileName}.png')

In [ ]:
train_path = "/Users/arjein/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/train.csv"
test_path = "/Users/arjein/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/test.csv"

# Load the dataset and assign column names
raw_data = pd.read_csv(train_path, header=None, names=['polarity', 'title', 'text'])

# Display the first few rows to verify
print(raw_data.head())
save_snapshot(raw_data,"raw_data")

In [ ]:
# Count missing values in each column
missing_title = raw_data['title'].isnull().sum()
missing_text = raw_data['text'].isnull().sum()

print(f"Missing values in 'title': {missing_title}")
print(f"Missing values in 'text': {missing_text}")


In [ ]:
import pandas as pd

def preprocess_sentiment_data(data):
    """
    Preprocess the sentiment data by:
    1. Mapping polarity values to sentiment labels (0 for negative, 1 for positive).
    2. Dropping rows where 'title' or 'text' is missing or empty.
    3. Returning the preprocessed dataframe with clean columns for further processing.

    Parameters:
    - data (pd.DataFrame): Input dataframe with 'polarity', 'title', and 'text' columns.

    Returns:
    - pd.DataFrame: Preprocessed dataframe with 'title', 'text', and 'sentiment' columns.
    """
    # Map polarity values to sentiment labels
    data['sentiment'] = data['polarity'].map({1: 0, 2: 1})
    
    # Drop rows where 'title' or 'text' is missing or empty
    data = data.dropna(subset=['title', 'text'])  # Drop rows where title or text is NaN
    data = data[data['title'].str.strip() != ""]  # Remove rows with empty title
    data = data[data['text'].str.strip() != ""]   # Remove rows with empty text
    
    # Drop unnecessary columns
    data = data.drop(columns=["polarity"])
    
    return data




In [ ]:
df = preprocess_sentiment_data(raw_data)
print(df.head())
save_snapshot(df, fileName="preprocess_sentiment")

## 2. Preprocessing the Data

To prepare the text for analysis, we will:
1. Convert the text to lowercase.
2. Remove punctuation and special characters.
3. Tokenize the text into individual words.
4. Remove stopwords like "the" and "is."
5. Apply stemming to reduce words to their root form.


In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Function to lowercase text
def lowercase_text(text):
    return text.lower()

# Function to remove punctuation and special characters
def remove_punctuation(text):
    return re.sub(r'[^a-z\s]', '', text)

# Function to tokenize text
def tokenize_text(text):
    return word_tokenize(text)

# Function to remove stopwords
def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [word for word in tokens if word not in stop_words]

# Function to stem tokens
def stem_tokens(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(word) for word in tokens]
    


In [ ]:
df['title'] = df['title'].apply(lowercase_text)
df['text'] = df['text'].apply(lowercase_text)
save_snapshot(df, fileName="lowercase")

In [ ]:
df['title'] = df['title'].apply(remove_punctuation)
df['text'] = df['text'].apply(remove_punctuation)

save_snapshot(df, fileName="remove_punctuation")

In [ ]:
# Tokenize the 'title' column
df['title_tokens'] = df['title'].apply(word_tokenize)

# Tokenize the 'text' column
df['text_tokens'] = df['text'].apply(word_tokenize)

save_snapshot(df, fileName="tokenized")

In [ ]:
df['title_tokens'] = df['title_tokens'].apply(remove_stopwords)
df['text_tokens'] = df['text_tokens'].apply(remove_stopwords)
print(df.loc[5])
save_snapshot(df, fileName="remove_stopwords")

In [ ]:
df['text_tokens'] = df['text_tokens'].apply(stem_tokens)
df['title_tokens'] = df['title_tokens'].apply(stem_tokens)
save_snapshot(df, fileName="stem_tokens")

In [ ]:
df['processed_title'] = df['title_tokens'].apply(lambda x: ' '.join(x))
df['processed_text'] = df['text_tokens'].apply(lambda x: ' '.join(x))


In [ ]:
# Save DataFrame to a CSV file
df.to_csv("amazon_review_preprocessed.csv", index=False)


## 3. Vectorizing the Text


In [ ]:
from sklearn.model_selection import train_test_split

# Define features and labels
X = df[['processed_title', 'processed_text']]  # Use both title and text columns
y = df['sentiment']  # Sentiment labels

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Initialize TF-IDF vectorizers for title and text
tfidf_title = TfidfVectorizer(max_features=1000)  # Limit title features
tfidf_text = TfidfVectorizer(max_features=5000)  # Limit text features

# Fit on the training data
X_train_title = tfidf_title.fit_transform(X_train['processed_title'])
X_train_text = tfidf_text.fit_transform(X_train['processed_text'])

# Transform the validation data
X_val_title = tfidf_title.transform(X_val['processed_title'])
X_val_text = tfidf_text.transform(X_val['processed_text'])


In [ ]:
# Combine title and text vectors
X_train_combined = hstack([X_train_title, X_train_text])
X_val_combined = hstack([X_val_title, X_val_text])

## 4. Training the Model


In [1]:
# Import data from CSV
import pandas as pd
df = pd.read_csv("amazon_review_preprocessed.csv")

In [2]:
X = df[['processed_title', 'processed_text']]  # Use both title and text columns
y = df['sentiment']  # Sentiment labels



In [3]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_train['processed_title'] = X_train['processed_title'].fillna("").astype(str)
X_train['processed_text'] = X_train['processed_text'].fillna("").astype(str)
X_val['processed_title'] = X_val['processed_title'].fillna("").astype(str)
X_val['processed_text'] = X_val['processed_text'].fillna("").astype(str)


from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Initialize TF-IDF vectorizers for title and text
tfidf_title = TfidfVectorizer(max_features=1000)  # Limit title features
tfidf_text = TfidfVectorizer(max_features=5000)  # Limit text features

# Fit on the training data
X_train_title = tfidf_title.fit_transform(X_train['processed_title'])
X_train_text = tfidf_text.fit_transform(X_train['processed_text'])

# Transform the validation data
X_val_title = tfidf_title.transform(X_val['processed_title'])
X_val_text = tfidf_text.transform(X_val['processed_text'])

# Combine title and text vectors
X_train_combined = hstack([X_train_title, X_train_text])
X_val_combined = hstack([X_val_title, X_val_text])

from sklearn.linear_model import LogisticRegression

# Initialize and train the model
model = LogisticRegression()
model.fit(X_train_combined, y_train)

from sklearn.metrics import accuracy_score, classification_report

# Predict on the validation set
y_val_pred = model.predict(X_val_combined)

# Print evaluation metrics
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Classification Report:\n", classification_report(y_val, y_val_pred))



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# Define parameter grids for each model
param_grid = [
    {
        'model': [LogisticRegression()],

    },
    {
            
        'model': [RandomForestClassifier()],
        
    },
    {
        
        
        'model': [SVC()],
        
    }
]


# Preprocessing pipeline for title and text
preprocessor = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(max_features=1000), 'processed_title'),
        ('text', TfidfVectorizer(max_features=5000), 'processed_text')
    ]
)

# Full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())  # Placeholder for the model, replaced during GridSearchCV
])

from sklearn.model_selection import GridSearchCV

# Initialize GridSearchCV
grid_search = GridSearchCV(
    pipeline,                # Pipeline object
    param_grid,              # Parameter grid
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',      # Metric to optimize
    verbose=3,               # Output verbosity
    n_jobs=-1                # Use all CPU cores for faster computation
)







In [ ]:
# Fit GridSearchCV on training data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END ........model=LogisticRegression();, score=0.890 total time= 9.9min


In [ ]:
# Get the best model and parameters
print("Best Model:", grid_search.best_estimator_)
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)


## 5. Evaluating the Model


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on the validation set
y_val_pred = model.predict(X_val_combined)

# Print evaluation metrics
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Classification Report:\n", classification_report(y_val, y_val_pred))


In [ ]:
y_train_pred = model.predict(X_train_combined)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
